In [1]:
%pip install instructor groq pydantic pandas tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sys
from dotenv import load_dotenv

# ADD THIS PART: Tell Python where to find the 'src' folder
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

load_dotenv(os.path.join(parent_dir, ".env"), override=True)
key = os.getenv("GROQ_API_KEY")

if key:
    print(f"Key Length: {len(key)}")
    print(f"Key starts with: {key[:7]}")
    from src.extraction.processor import EnronProcessor
    processor = EnronProcessor(api_key=key)
    print("Processor initialized successfully!")
else:
    print("Key is still missing!")

Key Length: 56
Key starts with: gsk_kqj
Processor initialized successfully!


In [3]:
# Force-write the .env file with correct formatting
with open("../.env", "w", encoding="utf-8") as f:
    f.write('GROQ_API_KEY=gsk_your_actual_key_here')

print(".env file rewritten from inside the notebook.")
print("Now go back and RE-RUN the Setup Cell (Cell 1).")

.env file rewritten from inside the notebook.
Now go back and RE-RUN the Setup Cell (Cell 1).


In [4]:
import pandas as pd
# Use the absolute path to your study set
path = r"c:\Users\hello\OneDrive\Documents\Intern\Layer10_Project\src\extraction\layer10_study_set.csv"
df = pd.read_csv(path)
print(f"Data Loaded {len(df)}")

Data Loaded 2000


In [5]:
# Force a clean write with no extra spaces
my_key = "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".strip()

with open("../.env", "w", encoding="utf-8") as f:
    f.write(f"GROQ_API_KEY={my_key}")

print(".env file cleaned and rewritten.")

.env file cleaned and rewritten.


In [6]:
import json

# Let's pick a specific email to test (index 1 is usually a good one)
sample_row = df.iloc[1] 
print(f"Processing: {sample_row['file']}")

try:
    # This calls your src code to talk to Groq
    result = processor.process_row(sample_row)
    
    if result:
        print("EXTRACTION SUCCESSFUL")
        # This converts the Pydantic object into a readable JSON format
        print(json.dumps(result.dict(), indent=2))
    else:
        print("Processor returned None")
except Exception as e:
    print(f"Extraction failed: {e}")

Processing: dasovich-j/notes_inbox/11956.
EXTRACTION SUCCESSFUL
{
  "entities": [
    {
      "name": "The Economist",
      "label": "ORG"
    },
    {
      "name": "John Mayo",
      "label": "PERSON"
    },
    {
      "name": "Marconi",
      "label": "ORG"
    },
    {
      "name": "AT&T",
      "label": "ORG"
    },
    {
      "name": "Compaq Computer",
      "label": "ORG"
    },
    {
      "name": "Microsoft",
      "label": "ORG"
    },
    {
      "name": "Baltimore Technologies",
      "label": "ORG"
    },
    {
      "name": "Webvan",
      "label": "ORG"
    },
    {
      "name": "Real Madrid",
      "label": "ORG"
    }
  ],
  "relationships": [
    {
      "source_entity": "The Economist",
      "target_entity": "Marconi",
      "relation_type": "MENTIONED_IN"
    },
    {
      "source_entity": "AT&T",
      "target_entity": "Comcast",
      "relation_type": "COMPETITOR"
    },
    {
      "source_entity": "Marconi",
      "target_entity": "John Mayo",
      "rela

C:\Users\hello\AppData\Local\Temp\ipykernel_3336\2435424433.py:14: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  print(json.dumps(result.dict(), indent=2))


In [7]:
import os
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

key = os.getenv("GROQ_API_KEY")

if key:
    # Check for quotes and spaces outside the f-string
    has_quote = key.startswith('"') or key.startswith("'")
    has_space = key.endswith(" ")
    
    print(f"Key Length: {len(key)}")
    print(f"First 4: {key[:4]}")
    print(f"Last 4: {key[-4:]}")
    print(f"Starts with quote? {has_quote}")
    print(f"Ends with space? {has_space}")
else:
    print("Key is None - Python still can't see the variable.")

Key Length: 56
First 4: gsk_
Last 4: HNXq
Starts with quote? False
Ends with space? False


In [8]:
import json
import os

batch = df.head(50)
all_results = []

print(f"⏳ Starting batch processing for {len(batch)} emails...")

for index, row in batch.iterrows():
    try:
        extraction = processor.process_row(row)
        
        if extraction:
            # Use model_dump() to avoid the deprecation warning
            data = extraction.model_dump()
            # Adding the original filename for traceability
            data['metadata'] = {"file": row['file']}
            all_results.append(data)
            
        if (index + 1) % 10 == 0:
            print(f"Processed {index + 1}/50...")
            
    except Exception as e:
        print(f"Error on {row['file']}: {e}")

# 2. Save the output to your data folder
output_path = os.path.join("..", "data", "extracted_memories.json")
with open(output_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"FINISHED {output_path}")

⏳ Starting batch processing for 50 emails...
❌ Error in kaminski-v/discussion_threads/1832.: <failed_attempts>

<generation number="1">
<exception>
    Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kk543ba1f84scxx4rsdksadb` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 957. Please try again in 13m46.848s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
</exception>
<completion>
    None
</completion>
</generation>

<generation number="2">
<exception>
    Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kk543ba1f84scxx4rsdksadb` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 957. Please try again in 13m46.848s. Need more tokens? Upgrade to Dev Tier today at https://console.gr